## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-24-10-53-37 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...
1,2026-11-24-10-51-00 +0000,wsj,The White House Push to Reinstate Jensen Huang,https://www.wsj.com/articles/the-white-house-p...
2,2026-11-24-10-50-00 +0000,wsj,Is Nvidia Becoming a Government-Sponsored Ente...,https://www.wsj.com/politics/policy/is-nvidia-...
3,2026-11-24-10-50-00 +0000,wsj,Jensen Huang’s return means Nvidia may now be ...,https://www.wsj.com/politics/policy/is-nvidia-...
4,2026-11-24-10-48-27 +0000,bbc,DR Congo says military aircraft missing as reb...,https://www.bbc.co.uk/news/articles/cwy7r5z8gg...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2613/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
10,huang,60
6,nvidia,50
9,jensen,46
0,trump,45
18,return,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
53,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...,277
0,2026-11-24-10-53-37 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,240
314,2026-11-23-15-53-43 +0000,nypost,Trump suggests Jensen Huang could have moved t...,https://nypost.com/2026/11/23/us-news/trump-su...,238
141,2026-11-23-23-46-05 +0000,nypost,Five wild Trump quotes about reinstated Nvidia...,https://nypost.com/2026/11/23/us-news/five-wil...,238
297,2026-11-23-17-17-17 +0000,nypost,Trump posts edited image of assassination atte...,https://nypost.com/2026/11/23/us-news/trump-po...,237


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
53,277,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...
375,54,2026-11-23-12-03-00 +0000,wsj,Five things to know about California Rep. Robe...,https://www.wsj.com/politics/policy/five-thing...
235,51,2026-11-23-20-13-46 +0000,nypost,Eric Swalwell’s pal Jimmy Gomez violated House...,https://nypost.com/2026/11/23/us-news/eric-swa...
374,49,2026-11-23-12-10-26 +0000,chitri,Ukrainian drones hit Russian warship at naval ...,https://www.chicagotribune.com/2026/11/23/ukra...
83,46,2026-11-24-04-15-00 +0000,wsj,A federal appeals court has cleared the way fo...,https://www.wsj.com/politics/national-security...
211,44,2026-11-23-21-24-51 +0000,nypost,Nasdaq surges 6.5% in second-largest one-day g...,https://nypost.com/2026/11/23/business/nasdaq-...
231,42,2026-11-23-20-30-16 +0000,nypost,Russia has lost astonishing 700K soldiers sinc...,https://nypost.com/2026/11/23/world-news/russi...
90,39,2026-11-24-03-30-00 +0000,nypost,China’s second-richest man’s deal to drain Cal...,https://nypost.com/2026/11/23/us-news/chinese-...
128,39,2026-11-24-00-31-28 +0000,bbc,No 10 North will face greater scrutiny as Burn...,https://www.bbc.co.uk/news/articles/c6rrg7jx5g...
89,38,2026-11-24-03-38-00 +0000,wsj,Ilhan Omar Investigation Covers Most of Her Li...,https://www.wsj.com/politics/policy/ilhan-omar...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
